# Indices de lexicalité des OCR générés

In [ ]:
#import duckdb
#con = duckdb.connect("../data/ocr-fulltext.db");
#con.sql("SELECT id, decode(content) FROM fulltext_ocr_generated LIMIT 100;").df()

In [ ]:
pip install wordfreq psycopg2-binary

In [ ]:
import re
import unicodedata
from wordfreq import zipf_frequency
#import psycopg2
#import sqlite3
from multiprocessing import Pool, cpu_count
import csv
import duckdb

# Procédure
- Tokenisation (extraction des mots composés de lettres : evite les formules, molécules)
- Normalisation (minuscules, accents)
- Comptabilisation des mots puis des mots valides (zipf_frequency > 2.5) et invalides
- Indice de lexicalité (valid_count / total_words)


In [ ]:
# Extrait uniquement les "mots alphabétiques"
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text)

In [ ]:
def normalize_text(text):
    # Minuscule
    text = text.lower()

    # Normalisation unicode (accents)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Supprime caractères non alphabétiques sauf espaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Nettoyage espaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# Calcule lexicalité et stats à partir d’un texte
from collections import Counter

def analyze_text(text, threshold=2.5, lang="en", top_n=20):
    normalized = normalize_text(text)
    tokens = tokenize(normalized)
    total_words = len(tokens)

    valid_words = []
    invalid_words = []

    for word in tokens:
        freq = zipf_frequency(word, lang)
        if freq >= threshold:
            valid_words.append(freq)
        else:
            invalid_words.append(word)

    valid_count = len(valid_words)
    invalid_count = len(invalid_words)
    lexicality = (valid_count / total_words * 100) if total_words else 0
    avg_zipf = sum(valid_words) / len(valid_words) if valid_words else 0

    # Top invalides sérialisés en string "mot:count|mot:count|..."
    top_invalid = Counter(invalid_words).most_common(top_n)
    top_invalid_str = "|".join(f"{w}:{c}" for w, c in top_invalid)

    return {
        "total_words": total_words,
        "valid_words": valid_count,
        "invalid_words": invalid_count,
        "lexicality": lexicality,
        "avg_zipf": avg_zipf,
        "top_invalid": top_invalid_str,
    }

In [ ]:
# Lit la base par blocs 
def stream_data(batch_size=1000):
    con = duckdb.connect("../data/ocr-fulltext.db")
    offset = 0
    while True:
        batch = con.sql(f"""
            SELECT id, try(decode(content)) AS content_text
            FROM fulltext_ocr_generated
            LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
        if not batch:
            break
        yield batch
        offset += batch_size

In [ ]:
# Décode le blob et analyse un document
def process_row(row, threshold=2.5, lang="en"):
    doc_id, content = row

    if content is None:
        return None

    # BLOB → bytes
    if isinstance(content, bytes):
        try:
            text = content.decode("utf-8", errors="ignore")
        except:
            # fallback si encodage exotique
            text = content.decode("latin-1", errors="ignore")
    else:
        text = str(content)

    analysis = analyze_text(text, threshold, lang)

    return {
        "id": doc_id,
        **analysis
    }

In [ ]:
# Applique multiprocessing sur les batches de données
def process_database_parallel(batch_size=1000, threshold=2.5, lang="en"):
    n_workers = max(cpu_count() - 1, 1)
    print(f"Workers utilisés: {n_workers}")

    results = []

    with Pool(n_workers) as pool:
        for batch in stream_data(batch_size):

            processed = pool.starmap(
                process_row,
                [(row, threshold, lang) for row in batch]
            )

            results.extend([r for r in processed if r is not None])

    return results

In [ ]:
# Agrège les résultats globaux
def compute_global_summary(report):
    total_docs = len(report)
    total_words = sum(r["total_words"] for r in report)
    total_valid = sum(r["valid_words"] for r in report)
    lexicality = (total_valid / total_words * 100) if total_words else 0

    # Agrège tous les invalides
    global_invalid = Counter()
    for r in report:
        if r.get("top_invalid"):
            for item in r["top_invalid"].split("|"):
                w, c = item.split(":")
                global_invalid[w] += int(c)

    return {
        "total_docs": total_docs,
        "total_words": total_words,
        "lexicality": lexicality,
        "top_invalid_global": global_invalid.most_common(50),
    }

In [ ]:
# Sauvegarde 
def export_report_to_csv(report, output_path="report.csv"):
    if not report:
        return

    fieldnames = report[0].keys()

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(report)

In [ ]:
# Lance tout le pipeline
BATCH_SIZE = 1000
THRESHOLD = 2.5
LANG = "en"

report = process_database_parallel(
    batch_size=BATCH_SIZE,
    threshold=THRESHOLD,
    lang=LANG
)

summary = compute_global_summary(report)

print("\n=== GLOBAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

export_report_to_csv(report, "ocr_lexicality-report-non-natif.csv")

In [ ]:
# Optionnel - exploration des résultats
#df = pd.read_csv("ocr_lexicality-report.csv")
#df["top_invalid_parsed"] = df["top_invalid"].apply(parse_top_invalid)

# Exemple : quels docs ont le plus de mots invalides ?
#df.sort_values("invalid_words", ascending=False).head(20)

In [ ]:
import pandas as pd
df = pd.read_csv("ocr_lexicality-report-non-natif.csv")

# Distribution par tranches
bins = [0, 50, 70, 85, 95, 100]
labels = ["<50%", "50-70%", "70-85%", "85-95%", ">95%"]
df["quality_tier"] = pd.cut(df["lexicality"], bins=bins, labels=labels)

print(df["quality_tier"].value_counts().sort_index())
print(df.groupby("quality_tier", observed=True)["total_words"].mean())

Interprétation :

In [ ]:
df["is_short"] = df["total_words"] < 500

print(df.groupby(["quality_tier", "is_short"], observed=True).size())

In [ ]:
# Les vrais ratés OCR : longs mais mauvaise lexicalité
priority = df[(df["total_words"] >= 500) & (df["lexicality"] < 50)]
print(f"{len(priority)} docs à retraiter en priorité")
priority[["id", "total_words", "lexicality", "avg_zipf"]].to_csv("priority_reprocess-non-natif.csv", index=False)

# Les cas limites longs : potentiellement récupérables
borderline = df[(df["total_words"] >= 500) & (df["lexicality"].between(50, 70))]
print(f"{len(borderline)} docs borderline")

Les docs bordelines ne sont pas en anglais ou contiennent des listes de substances, des index...

In [ ]:
print(df.columns)
print(len(df))
print(df.head())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ── Chargement ────────────────────────────────────────────────────────
df = pd.read_csv("ocr_lexicality-report-non-natif.csv")

# ── Style (cohérent avec les graphes précédents du notebook) ──────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.edgecolor":   "#cccccc",
    "axes.labelcolor":  "#333333",
    "xtick.color":      "#555555",
    "ytick.color":      "#555555",
    "text.color":       "#111111",
    "grid.color":       "#e0e0e0",
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(figsize=(12, 6))

# ── Données ───────────────────────────────────────────────────────────
scores = df["lexicality"].dropna()
n = len(scores)

# Jitter horizontal pour éviter la superposition
rng = np.random.default_rng(42)
x = rng.normal(0, 0.15, size=n)

norm = Normalize(vmin=scores.min(), vmax=scores.max())
cmap = plt.cm.viridis

sc = ax.scatter(
    x, scores,
    c=scores, cmap=cmap,
    alpha=0.25, s=6,
    linewidths=0, rasterized=True,
)

# ── Statistiques superposées ──────────────────────────────────────────
q25, median, q75 = scores.quantile([0.25, 0.50, 0.75])
mean_val = scores.mean()

# positions horizontales (coordonnées axes)
x_left = 0.02
x_center = 0.50
x_right = 0.98

# Médiane (gauche)
ax.axhline(median, color="#333333", linewidth=1.4, linestyle="-", alpha=0.8)
ax.text(x_left, median + 0.8,
        f"Médiane {median:.1f} %",
        transform=ax.get_yaxis_transform(),
        ha="left", fontsize=9, color="#333333", va="bottom")

# Moyenne (centre)
ax.axhline(mean_val, color="#333333", linewidth=1.4, linestyle="--", alpha=0.8)
ax.text(x_center, mean_val + 0.8,
        f"Moyenne {mean_val:.1f} %",
        transform=ax.get_yaxis_transform(),
        ha="center", fontsize=9, color="#333333", va="bottom")

# Quantiles (droite)
ax.text(x_right, q25 - 0.8,
        f"Q1 = {q25:.1f} %",
        transform=ax.get_yaxis_transform(),
        ha="right", fontsize=8, color="#666666", va="top")

ax.text(x_right, q75 + 0.8,
        f"Q3 = {q75:.1f} %",
        transform=ax.get_yaxis_transform(),
        ha="right", fontsize=8, color="#666666", va="bottom")

# ── Colorbar ──────────────────────────────────────────────────────────
cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.03)
cbar.set_label("Score de lexicalité (%)", fontsize=10)
cbar.ax.tick_params(labelsize=8)

# ── Axes & titres ─────────────────────────────────────────────────────
ax.set_xlim(-0.6, 0.6)
ax.set_ylim(-2, 105)
ax.set_xticks([])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax.set_ylabel("Score de lexicalité", fontsize=11)
ax.set_title(
    "Dispersion des scores de lexicalité\n"
    f"(n = {n:,} documents — seuil Zipf ≥ 2.5)",
    fontsize=13, weight="bold", pad=14,
)
ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("fig_dispersion_lexicalite-non-natif.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : fig_dispersion_lexicalite-non-natif.png")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Données ───────────────────────────────────────────────────────────
df = pd.read_csv("ocr_lexicality-report-non-natif.csv")

# Tranches logarithmiques (plus lisibles pour des données skewed)
bins   = [0, 100, 500, 1_000, 2_500, 5_000, 10_000, 50_000, int(df["total_words"].max()) + 1]
labels = ["<100", "100–500", "500–1k", "1k–2.5k", "2.5k–5k", "5k–10k", "10k–50k", "50k+"]

df["word_bin"] = pd.cut(df["total_words"], bins=bins, labels=labels, right=False)

grp = df.groupby("word_bin", observed=True)["lexicality"].agg(
    median="median", q25=lambda s: s.quantile(0.25),
    q75=lambda s: s.quantile(0.75), count="count"
).reset_index()

x = np.arange(len(grp))
cmap = plt.cm.viridis
colors = [cmap(v) for v in np.linspace(0.1, 0.9, len(grp))]

# ── Figure ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Bande IQR (Q1–Q3)
ax.fill_between(x, grp["q25"], grp["q75"],
                alpha=0.18, color=cmap(0.55), label="Intervalle Q1–Q3")

# Ligne médiane principale
ax.plot(x, grp["median"],
        marker="o", linewidth=2.5, color=cmap(0.75),
        markerfacecolor="white", markeredgecolor=cmap(0.75),
        markeredgewidth=2, markersize=8,
        label="Lexicalité médiane (%)", zorder=5)

# Ligne Q1 & Q3 pointillées légères
for col, ls, alpha in [("q25", "--", 0.5), ("q75", "--", 0.5)]:
    ax.plot(x, grp[col], linestyle=ls, linewidth=1, color=cmap(0.45), alpha=alpha)

# Barres de volume (axe secondaire)
ax2 = ax.twinx()
ax2.bar(x, grp["count"], color=cmap(0.3), alpha=0.18, width=0.6, zorder=0)
ax2.set_ylabel("Nombre de documents", fontsize=10, color="#555555")
ax2.tick_params(axis="y", labelcolor="#555555", labelsize=8)
ax2.spines["top"].set_visible(False)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

# Annotations : médiane sur chaque point
for xi, row in zip(x, grp.itertuples()):
    ax.annotate(
        f"{row.median:.0f} %",
        xy=(xi, row.median),
        xytext=(0, 9), textcoords="offset points",
        ha="center", fontsize=8, color="#333333",
        fontweight="bold",
    )

# ── Axes & style ──────────────────────────────────────────────────────
ax.set_xticks(x)
ax.set_xticklabels(grp["word_bin"], fontsize=9)
ax.set_xlabel("Nombre de mots par document", fontsize=11)
ax.set_ylabel("Score de lexicalité (%)", fontsize=11)
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f %%"))
ax.grid(axis="y", alpha=0.35, color="#dddddd")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_title(
    "Lexicalité selon le nombre de mots\n"
    "Médiane par tranche — bande Q1/Q3 — volume en arrière-plan",
    fontsize=13, weight="bold", pad=14,
)
ax.legend(loc="lower right", fontsize=9, framealpha=0.7)

plt.tight_layout()
plt.savefig("fig_lexicalite_par_nb_mots-non-natif.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : fig_lexicalite_par_nb_mots-non-natif.png")
